[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/milvus-io/bootcamp/blob/master/bootcamp/RAG/search_with_jev/rerank_graph_relations.ipynb)
[![View on GitHub](https://img.shields.io/badge/View_on-GitHub-181717?logo=github)](https://github.com/milvus-io/bootcamp/blob/master/bootcamp/RAG/search_with_jev/rerank_graph_relations.ipynb)

# Rerank graph relations with Jev

Retrieve relationship records from Milvus, score both direct evidence and useful bridge relations, and carry the relationship order into the source-document order.

## Preparation

Run locally from this directory with `uv sync --python 3.12` and `uv run jupyter lab`, or install the notebook dependencies in Colab:

In [1]:
# In Colab, uncomment this setup cell. Local users should use uv sync --python 3.12.
# %pip install "pymilvus>=2.5,<2.6.10" "milvus-lite>=2.5,<3" "setuptools<71" "google-genai>=1.68,<2" numpy requests

> In Colab, restart the runtime after installing dependencies if needed.

Set `GEMINI_API_KEY` and `TYPESAFE_API_KEY` in your environment or enter them privately below. A `GOOGLE_API_KEY` is also accepted for Gemini. Obtain a Gemini key from [Google AI Studio](https://aistudio.google.com/apikey) and a TypeSafe key from the [TypeSafe console](https://console.typesafe.ai/). Gemini embeds the synthetic documents and queries; TypeSafe receives the sample evidence and judgment questions. Both services require API access and may consume credits.

The helper batches independent questions into one request. IDs map responses back to code; the instructions explicitly identify each field being judged. HTTP failures stop the tutorial rather than produce fabricated scores.

In [2]:
import getpass
import json
import math
import os
import time
import uuid

import requests
from pymilvus import DataType, MilvusClient
import numpy as np
from google import genai
from google.genai import types

if not os.getenv("TYPESAFE_API_KEY"):
    os.environ["TYPESAFE_API_KEY"] = getpass.getpass("TypeSafe API key: ")

if not os.getenv("GEMINI_API_KEY"):
    os.environ["GEMINI_API_KEY"] = os.getenv("GOOGLE_API_KEY") or getpass.getpass(
        "Gemini API key: "
    )

MODEL = os.getenv("JEV_MODEL", "jev-1.13.0")
API_URL = "https://api.typesafe.ai/v1/systemone"
call_log = []

### Call Jev

This helper sends independent questions in one request and validates the returned answers.

In [3]:
def judge(state, questions):
    """Call Jev with bounded retries; stop on invalid or incomplete responses."""
    for attempt in range(3):
        started = time.perf_counter()
        response = requests.post(
            API_URL,
            headers={"Authorization": f"Bearer {os.environ['TYPESAFE_API_KEY']}"},
            json={"model": MODEL, "state": state, "questions": questions},
            timeout=45,
        )
        if response.status_code in (429, 500, 502, 503, 504) and attempt < 2:
            time.sleep(2**attempt)
            continue
        response.raise_for_status()
        body = response.json()
        answers = body["answers"]
        if set(answers) != set(questions):
            raise ValueError("Jev returned missing or unexpected question IDs")
        for key, question in questions.items():
            answer = answers[key]
            if question["type"] == "noul":
                value = float(answer["noul"])
                if not math.isfinite(value) or not 0 <= value <= 1:
                    raise ValueError("Invalid Noul probability")
            elif answer["choice"] not in question["criteria"]:
                raise ValueError("Unexpected Choice option")
        call_log.append(
            {
                "seconds": round(time.perf_counter() - started, 3),
                "usage": body.get("usage", {}),
                "model": MODEL,
            }
        )
        return answers
    raise RuntimeError("Jev request failed")


def noul(instructions):
    return {
        "type": "noul",
        "instructions": instructions,
        "criteria": {
            "true": "The stated condition is supported by the supplied data.",
            "false": "The condition is unsupported or contradicted.",
        },
    }

## Prepare a small corpus

All names and records below are synthetic teaching examples.

In [4]:
relation_rows = [
    (1, "book_amber", "author_mira", "Amber Harbor -> written by -> Mira Sol", "The novel Amber Harbor was written by Mira Sol."),
    (2, "author_mira", "city_nacre", "Mira Sol -> born in -> Nacre", "Mira Sol was born in Nacre."),
    (3, "book_amber", "publisher_lantern", "Amber Harbor -> published by -> Lantern Press", "Amber Harbor was published by Lantern Press."),
    (4, "author_mira", "award_reed", "Mira Sol -> won -> Silver Reed", "Mira Sol received the Silver Reed literary award."),
    (5, "book_winter", "author_theo", "Winter Orchard -> written by -> Theo Vale", "Winter Orchard was written by Theo Vale."),
    (6, "author_theo", "city_brindle", "Theo Vale -> born in -> Brindle", "Theo Vale was born in Brindle."),
]
documents = [
    {"id": doc_id, "source_entity": source, "target_entity": target,
     "relation": relation, "text": text}
    for doc_id, source, target, relation, text in relation_rows
]

## Connect to Milvus

For `MilvusClient`:

- Use a local file such as `./search_with_jev.db` for [Milvus Lite](https://milvus.io/docs/milvus_lite.md).
- Set `MILVUS_URI` to a server endpoint such as `http://localhost:19530` for [Milvus on Docker or Kubernetes](https://milvus.io/docs/quickstart.md).
- For [Zilliz Cloud](https://zilliz.com/cloud), set `MILVUS_URI` to the public endpoint and `MILVUS_TOKEN` to your API key.

Each run uses its own collection name. Cleanup removes only that collection.

In [5]:
client = MilvusClient(
    uri=os.getenv("MILVUS_URI", "./search_with_jev.db"),
    token=os.getenv("MILVUS_TOKEN", ""),
)
collection_name = "jev_demo_" + uuid.uuid4().hex[:12]

## Encode the sample documents

Use [Gemini Embedding 2](https://ai.google.dev/gemini-api/docs/embeddings) to generate 768-dimensional semantic vectors. Milvus stores these vectors and retrieves candidates by cosine similarity; Jev judges the retrieved text afterward.

For this model, the retrieval task is specified in the input text, not the API's `task_type` field. Documents use `title: none | text: ...`, while queries use `task: search result | query: ...`. Each document is embedded separately because passing multiple inputs to Embedding 2 can aggregate them into one vector. The model normalizes its 768-dimensional output automatically.

Keep the model, dimension and formatting consistent between indexing and searching. If you change the embedding configuration, regenerate the document vectors and recreate the collection. These tiny examples fit within the model's input limit; split longer source documents into chunks before embedding them.

In [6]:
EMBEDDING_MODEL = "gemini-embedding-2"
EMBEDDING_DIMENSION = 768
embedding_client = genai.Client(
    api_key=os.environ["GEMINI_API_KEY"],
    http_options=types.HttpOptions(
        timeout=60000,
        retry_options=types.HttpRetryOptions(attempts=3),
    ),
)


def embed_text(text):
    """Embed one input, validating the vector before storing or searching."""
    result = embedding_client.models.embed_content(
        model=EMBEDDING_MODEL,
        contents=text,
        config=types.EmbedContentConfig(
            output_dimensionality=EMBEDDING_DIMENSION,
        ),
    )
    if not result.embeddings or len(result.embeddings) != 1:
        raise ValueError("Expected exactly one embedding per input")
    vector = np.asarray(result.embeddings[0].values, dtype=np.float32)
    if vector.shape != (EMBEDDING_DIMENSION,) or not np.isfinite(vector).all():
        raise ValueError("Invalid embedding dimension or values")
    if not np.linalg.norm(vector):
        raise ValueError("Received a zero embedding")
    return vector


# Embed documents separately: Embedding 2 can aggregate multiple inputs.
vectors = np.stack(
    [embed_text(f"title: none | text: {row['text']}") for row in documents]
)
print(f"Embedded {len(vectors)} documents with {EMBEDDING_MODEL}: {vectors.shape}")

Embedded 6 documents with gemini-embedding-2: (6, 768)


## Create the collection

Define the primary key, vector and text fields explicitly. Additional sample metadata is stored in dynamic fields. The vector index and search both use cosine similarity.

In [7]:
schema = client.create_schema(auto_id=False, enable_dynamic_field=True)
schema.add_field(field_name="id", datatype=DataType.INT64, is_primary=True)
schema.add_field(
    field_name="vector", datatype=DataType.FLOAT_VECTOR, dim=vectors.shape[1]
)
schema.add_field(field_name="text", datatype=DataType.VARCHAR, max_length=8192)
index_params = client.prepare_index_params()
index_params.add_index(
    field_name="vector", index_type="AUTOINDEX", metric_type="COSINE"
)
if not client.has_collection(collection_name=collection_name):
    client.create_collection(
        collection_name=collection_name,
        schema=schema,
        index_params=index_params,
        # consistency_level="Strong",
    )

## Insert the documents

Write the document text, metadata and vectors to Milvus.

In [8]:
client.insert(
    collection_name=collection_name,
    data=[dict(row, vector=vector.tolist()) for row, vector in zip(documents, vectors)],
)

{'insert_count': 6, 'ids': [1, 2, 3, 4, 5, 6], 'cost': 0}

## Retrieve candidates

Return only text and sample metadata, keeping vectors out of the Jev request. Strong consistency makes newly inserted documents searchable immediately.

In [9]:
output_fields = sorted({key for row in documents for key in row if key != "vector"})


def retrieve(query, limit=5, filter_expr=""):
    vector = embed_text(f"task: search result | query: {query}")
    hits = client.search(
        collection_name=collection_name,
        data=[vector.tolist()],
        anns_field="vector",
        limit=limit,
        filter=filter_expr,
        output_fields=output_fields,
        search_params={"metric_type": "COSINE", "params": {}},
        consistency_level="Strong",
    )[0]
    return [
        dict(
            {key: value for key, value in hit["entity"].items() if key != "vector"},
            id=hit["id"],
            retrieval_score=hit["distance"],
        )
        for hit in hits
    ]

## Retrieve seed relations, then expand one graph hop

Ask where the author of Amber Harbor was born. Search using the original question, without naming the author in advance. Retrieve two seed relations, read their entity IDs, then fetch relationships touching those entities with a Milvus metadata filter. The structured relationships are supplied as a tiny prepared graph; entity extraction is outside this tutorial.

Seed relations retain their vector order. Newly discovered neighbors follow in ID order for deterministic display; they do not have a vector retrieval rank. One expansion hop is enough to illustrate how an author-to-birthplace relationship can enter the shortlist. A production graph needs explicit traversal and candidate budgets.

In [10]:
query = "Where was the author of Amber Harbor born?"
seeds = retrieve(query, limit=2)
entity_ids = sorted({row[field] for row in seeds
                     for field in ("source_entity", "target_entity")})
entity_values = json.dumps(entity_ids)
neighbors = client.query(
    collection_name=collection_name,
    filter=f"source_entity in {entity_values} or target_entity in {entity_values}",
    output_fields=output_fields,
    consistency_level="Strong",
)
by_id = {row["id"]: row for row in seeds}
for row in sorted(neighbors, key=lambda row: row["id"]):
    by_id.setdefault(row["id"], row)
candidates = list(by_id.values())
seed_ids = {row["id"] for row in seeds}
print("Question:", query)
print("Seed IDs:", [row["id"] for row in seeds])
print("Expanded entity IDs:", entity_ids)
print(f"{len(documents)} graph relations -> {len(seeds)} seeds -> {len(candidates)} candidates")

Question: Where was the author of Amber Harbor born?
Seed IDs: [3, 1]
Expanded entity IDs: ['author_mira', 'book_amber', 'publisher_lantern']
6 graph relations -> 2 seeds -> 4 candidates


## Rerank useful relationships, including bridges

Ask one Noul question per candidate. A relationship can help by directly supplying the requested fact or by linking the question's entity to it. For example, in a film-director-nationality question, identifying the director is a useful bridge, whereas naming the distributor is not. This example explains the general rule without supplying the current question's answer.

In [11]:
answers = judge(
    {"query": query, "relations": candidates},
    {
        f"relation_{i}": noul(
            f"Candidate relation: {row['relation']}. Does this relationship help "
            "answer `query`, either directly or as a necessary bridge in a path "
            "supported by the supplied relations? A relationship about another "
            "entity with no supported connection is insufficient. For a film's "
            "director's nationality, film -> director and director -> nationality "
            "help, while film -> distributor does not. Use only the supplied facts."
        )
        for i, row in enumerate(candidates)
    },
)
ranked = sorted(
    [dict(row, candidate_position=i + 1, jev_score=answers[f"relation_{i}"]["noul"])
     for i, row in enumerate(candidates)],
    key=lambda row: row["jev_score"], reverse=True,
)
THRESHOLD = 0.5
selected = [row for row in ranked if row["jev_score"] >= THRESHOLD]

Show the candidate order, how each relation entered the shortlist, and the final score order. Equal scores preserve candidate order.

In [12]:
from IPython.display import Markdown, display


def show_table(headers, rows):
    def escape(value):
        return str(value).replace("|", "&#124;").replace("\n", " ")
    lines = ["| " + " | ".join(headers) + " |",
             "| " + " | ".join(["---"] * len(headers)) + " |"]
    lines.extend("| " + " | ".join(escape(v) for v in row) + " |" for row in rows)
    display(Markdown("\n".join(lines)))

show_table(
    ["Relation", "Origin", "Candidate position", "Jev rank", "Score", "Decision"],
    [(row["relation"], "Vector seed" if row["id"] in seed_ids else "Graph neighbor",
      row["candidate_position"], rank, f"{row['jev_score']:.3f}",
      "Keep" if row["jev_score"] >= THRESHOLD else "Exclude")
     for rank, row in enumerate(ranked, start=1)],
)

| Relation | Origin | Candidate position | Jev rank | Score | Decision |
| --- | --- | --- | --- | --- | --- |
| Amber Harbor -> written by -> Mira Sol | Vector seed | 2 | 1 | 0.980 | Keep |
| Mira Sol -> born in -> Nacre | Graph neighbor | 3 | 2 | 0.980 | Keep |
| Amber Harbor -> published by -> Lantern Press | Vector seed | 1 | 3 | 0.060 | Exclude |
| Mira Sol -> won -> Silver Reed | Graph neighbor | 4 | 4 | 0.050 | Exclude |

## Fetch source passages in the selected relation order

Each toy relation has one source passage. Fetch by ID, then explicitly reconstruct the chosen order because a database get operation need not preserve the input ID order. Relevance rank is not necessarily the traversal order of the graph path.

In [13]:
source_ids = list(dict.fromkeys(row["id"] for row in selected))
source_map = {
    row["id"]: row for row in client.get(
        collection_name=collection_name, ids=source_ids, output_fields=["text"],
        consistency_level="Strong",
    )
} if source_ids else {}
ordered_sources = [source_map[source_id] for source_id in source_ids]
print("Selected relation/source IDs:", source_ids)
for row in ordered_sources:
    print(f"[{row['id']}] {row['text']}")

Selected relation/source IDs: [1, 2]
[1] The novel Amber Harbor was written by Mira Sol.
[2] Mira Sol was born in Nacre.


The useful chain in this constructed graph is Amber Harbor -> Mira Sol -> Nacre. Check whether both edges entered the candidate set and survived the gate; reranking cannot recover a missing edge. Keeping the chain demonstrates selection, not a benchmark improvement. No final answer is generated here.

## Inspect usage and clean up

The raw usage fields and request duration help inspect this run. They are not a latency benchmark.

In [14]:
print(json.dumps(call_log, indent=2))
client.drop_collection(collection_name=collection_name)
client.close()
embedding_client.close()

[
  {
    "seconds": 0.664,
    "usage": {
      "input_tokens": 1064,
      "output_tokens": 76
    },
    "model": "jev-1.13.0"
  }
]


## Next steps

The thresholds in this example are starting points, not calibrated production defaults. Independent questions share state but do not see each other's answers. See the [Jev primitives](https://docs.typesafe.ai/primitives) and the [cookbook index](https://github.com/milvus-io/bootcamp/blob/master/bootcamp/RAG/search_with_jev/README.md).

For a larger example, see Vector Graph RAG's [Jev implementation](https://github.com/zilliztech/vector-graph-rag/blob/main/src/vector_graph_rag/llm/jev.py) and [evaluation](https://github.com/zilliztech/vector-graph-rag/blob/main/evaluation/jev/README.md).